In [ ]:
# last_verified: 2026-10-02 · Checkov n/a
# Rule class under comparison: "no S3 bucket may carry a public ACL".
# The four strategies below encode the decision rule of four custom-policy
# authoring styles so they can be scored against one labelled corpus.

# Checkov custom policy comparison patterns — four authoring styles for one rule

Every organisation eventually writes a Checkov custom policy for a rule the
built-in checks do not express. The rule is rarely the hard part. The hard part
is that the same rule can be expressed in several authoring styles, and those
styles differ in ways that are invisible until a resource is configured slightly
differently than expected.

This notebook takes one rule class — *no S3 bucket may carry a public ACL* — and
expresses it four ways: a single-attribute YAML policy, a whole-value YAML policy
set, a per-key YAML policy set, and a single Python check. Each strategy is scored
against the same labelled fixture corpus for recall, precision, and false
positives, then the resulting split is inspected resource by resource.

The strategies are evaluated by a local harness rather than by invoking Checkov,
so the notebook runs anywhere and the arithmetic is reproducible. The wiring that
hands the same four policies to Checkov in a scan is shown separately.

## Purpose

Establish which custom-policy authoring style to reach for, and why, on a
measured basis rather than by preference. The notebook covers:

- The labelled fixture corpus every strategy is scored against.
- Four strategies for the same rule, each with its decision rule stated.
- Recall, precision, and false positives per strategy, printed as one table.
- The resulting split inspected resource by resource to explain the numbers.
- A verification cell asserting the expected split, so a later edit that changes
  a strategy's semantics fails loudly instead of silently re-scoring.
- The Checkov wiring for each style, for use in a scan.

## When to use

- **Single-attribute policy** — the exposure is expressed as one scalar attribute
  on one resource type, and that is the only surface you intend to police.
- **Whole-value policy set** — never, on its own. It is included here as the
  baseline whose failure mode motivates the next two strategies.
- **Per-key policy set** — the rule spans a small, known, stable set of surfaces
  and the team prefers declarative YAML that reviewers can read without running
  Python.
- **Python check** — the rule spans surfaces that may grow, the pass condition is
  not a single comparison, or the report needs to distinguish "protected" from
  "carries no ACL information at all".

## Prerequisites

- Python 3.9 or newer to run the harness. No Checkov installation is required for
  the comparison; the wiring cells are reference material.
- A working understanding of how Terraform expresses a bucket's ACL: either
  inline on `aws_s3_bucket` as an `acl` attribute, or out-of-band on a separate
  `aws_s3_bucket_public_access_block` resource.

## The rule under comparison

> No S3 bucket may carry a public ACL. Absence of ACL information is **not** a
> finding — a bucket with no `acl` attribute and no public-access block carries
> no public ACL, so it is not exposed.

That second sentence is what separates the strategies. Each one has to make a
decision about resources that say nothing about their ACL state, and the four
handle it differently.

The first strategy is the one already in this kit, at
`../policies/no-public-s3-buckets/no_public_s3_buckets.yaml` — a single
attribute comparison in the `metadata` / `scope` / `definition` shape:

```yaml
metadata:
  id: CKV_CUSTOM_1
  name: "Ensure S3 buckets do not have public ACLs"
  category: general
scope:
  provider: aws
definition:
  cond_type: attribute
  resource_types:
    - aws_s3_bucket
  attribute: acl
  operator: not_equals
  value: "private"
```

It is precise, cheap, and reviewable. It is also scoped to exactly one surface.

## Step 1 — Build the labelled corpus

Twelve resources, each with a ground-truth `exposed` label. The set deliberately
includes the shapes that separate the strategies: the exposure carried by the
inline `acl` attribute, the exposure carried by an inline
`public_access_block` block, the same exposure carried by the standalone
resource type, a bucket that configures only some of the block's keys, a
standalone resource that relaxes the *policy* surface while leaving the ACL
surface locked, and a bucket that says nothing at all.

In [ ]:
FIXTURES = [
    # --- exposed -----------------------------------------------------------
    {"address": "aws_s3_bucket.logs", "type": "aws_s3_bucket",
     "config": {"acl": "public-read"}, "exposed": True},
    {"address": "aws_s3_bucket.legacy_site", "type": "aws_s3_bucket",
     "config": {"acl": "public-read-write"}, "exposed": True},
    {"address": "aws_s3_bucket.inline_block", "type": "aws_s3_bucket",
     "config": {"public_access_block": {"block_public_acls": False,
                                        "block_public_policy": True,
                                        "ignore_public_acls": True,
                                        "restrict_public_buckets": True}},
     "exposed": True},
    {"address": "aws_s3_bucket_public_access_block.acls_open",
     "type": "aws_s3_bucket_public_access_block",
     "config": {"block_public_acls": False}, "exposed": True},
    {"address": "aws_s3_bucket_public_access_block.all_open",
     "type": "aws_s3_bucket_public_access_block",
     "config": {"block_public_acls": False, "block_public_policy": False,
                "ignore_public_acls": False, "restrict_public_buckets": False},
     "exposed": True},

    # --- not exposed -------------------------------------------------------
    {"address": "aws_s3_bucket.private_acl", "type": "aws_s3_bucket",
     "config": {"acl": "private"}, "exposed": False},
    {"address": "aws_s3_bucket.fully_locked", "type": "aws_s3_bucket",
     "config": {"public_access_block": {"block_public_acls": True,
                                        "block_public_policy": True,
                                        "ignore_public_acls": True,
                                        "restrict_public_buckets": True}},
     "exposed": False},
    {"address": "aws_s3_bucket.partial_block", "type": "aws_s3_bucket",
     "config": {"public_access_block": {"block_public_acls": True}}, "exposed": False},
    {"address": "aws_s3_bucket.unrelated_block_keys", "type": "aws_s3_bucket",
     "config": {"public_access_block": {"ignore_public_acls": True,
                                        "restrict_public_buckets": True}},
     "exposed": False},
    {"address": "aws_s3_bucket.bare", "type": "aws_s3_bucket",
     "config": {}, "exposed": False},
    {"address": "aws_s3_bucket_public_access_block.fully_blocked",
     "type": "aws_s3_bucket_public_access_block",
     "config": {"block_public_acls": True, "block_public_policy": True,
                "ignore_public_acls": True, "restrict_public_buckets": True},
     "exposed": False},
    {"address": "aws_s3_bucket_public_access_block.policies_open",
     "type": "aws_s3_bucket_public_access_block",
     "config": {"block_public_acls": True, "block_public_policy": False},
     "exposed": False},
]

EXPOSED = {f["address"] for f in FIXTURES if f["exposed"]}
COMPLIANT = {f["address"] for f in FIXTURES if not f["exposed"]}

print(f"corpus: {len(FIXTURES)} resources, {len(EXPOSED)} exposed, {len(COMPLIANT)} not exposed")

## Step 2 — Strategy A: one attribute comparison

`attribute: acl`, `operator: not_equals`, `value: "private"`. Two decisions are
baked into it: the resource type is `aws_s3_bucket`, and the only attribute
consulted is `acl`.

A resource whose `acl` attribute is absent cannot be evaluated by the policy, so
it passes. That is the correct outcome for this corpus — but it is correct by
omission, not by reasoning, and it is the same behaviour that would let a real
exposure through on a surface the policy never names.

In [ ]:
def strategy_a(resources):
    """Single-attribute policy: aws_s3_bucket, `acl` not_equals `private`.

    Returns {address: [evaluated keys]}. An absent attribute is not evaluable, so
    the resource is not reported.
    """
    hits = {}
    for res in resources:
        keys = []
        if res["type"] == "aws_s3_bucket":
            acl = res["config"].get("acl")
            if acl is not None and acl != "private":
                keys.append("acl")
        if keys:
            hits[res["address"]] = keys
    return hits


for address, keys in strategy_a(FIXTURES).items():
    print(f"  FAIL {address:52s} evaluated {keys}")

## Step 3 — Strategy B: whole-value policies

The obvious next step is to cover the other surfaces with policies of the same
shape. The temptation is to compare the entire `public_access_block` value
against the fully-locked shape: one comparison per resource type, and no need to
know which key inside the block actually matters.

That is Strategy B, and it over-reports. A block that sets only the keys it
cares about is not equal to the fully-locked block, so it is reported — even
when every key it did set is safe. The comparison cannot tell *which* key differs,
only that the values differ.

In [ ]:
FULLY_LOCKED = {"block_public_acls": True, "block_public_policy": True,
                "ignore_public_acls": True, "restrict_public_buckets": True}


def strategy_b(resources):
    """Whole-value policy set: three policies, each comparing a whole attribute
    value against a literal — `acl` against `private`, each block-shaped value
    against the fully-locked shape.

    Returns {address: [evaluated keys]}.
    """
    hits = {}
    for res in resources:
        keys = []
        if res["type"] == "aws_s3_bucket":
            acl = res["config"].get("acl")
            if acl is not None and acl != "private":
                keys.append("acl")
            block = res["config"].get("public_access_block")
            if block is not None and block != FULLY_LOCKED:
                keys.append("public_access_block")
        elif res["type"] == "aws_s3_bucket_public_access_block":
            if res["config"] != FULLY_LOCKED:
                keys.append("<resource body>")
        if keys:
            hits[res["address"]] = keys
    return hits


for address, keys in strategy_b(FIXTURES).items():
    verdict = "false positive" if address in COMPLIANT else "true positive"
    print(f"  FAIL {address:52s} evaluated {keys}  <- {verdict}")

## Step 4 — Strategy B′: one policy per key

The fix for Strategy B is to stop comparing whole blocks and compare the single
key that governs the ACL surface. `block_public_acls` is the key that decides
whether a public ACL can take effect; `block_public_policy` governs a different
surface entirely and is not part of this rule.

Split into one policy per surface, each targeting that one key, Strategy B′ loses
the over-reporting. It is still declarative YAML, still reviewable without
running anything.

In [ ]:
def strategy_b_prime(resources):
    """Per-key policy set: one policy per surface, each targeting the single
    boolean that governs the ACL surface, plus the inline `acl` attribute.

    Returns {address: [evaluated keys]}.
    """
    hits = {}
    for res in resources:
        keys = []
        if res["type"] == "aws_s3_bucket":
            acl = res["config"].get("acl")
            if acl is not None and acl != "private":
                keys.append("acl")
            block = res["config"].get("public_access_block", {})
            if block.get("block_public_acls") is False:
                keys.append("public_access_block.block_public_acls")
        elif res["type"] == "aws_s3_bucket_public_access_block":
            if res["config"].get("block_public_acls") is False:
                keys.append("block_public_acls")
        if keys:
            hits[res["address"]] = keys
    return hits


for address, keys in strategy_b_prime(FIXTURES).items():
    verdict = "false positive" if address in COMPLIANT else "true positive"
    print(f"  FAIL {address:52s} evaluated {keys}  <- {verdict}")

## Step 5 — Strategy C: one Python check

The Python form collapses the surface enumeration into one place and, more
usefully, returns a three-state classification rather than a binary verdict:
`exposed`, `protected`, or `unclassified`. A binary check cannot express the
difference between a bucket that locked its ACL surface and a bucket that never
mentions it — both are "not a failure" — and that difference is the one a team
usually wants to report on.

In [ ]:
def classify(res):
    """Return 'exposed', 'protected', or 'unclassified' for one resource.

    'unclassified' means the resource carries no ACL information on any surface
    this check knows about. It is neither a pass nor a finding — it is a gap in
    what the policy can see.
    """
    rtype, config = res["type"], res["config"]
    if rtype == "aws_s3_bucket":
        acl = config.get("acl")
        block = config.get("public_access_block")
        if acl is None and block is None:
            return "unclassified"
        if acl is not None and acl != "private":
            return "exposed"
        if isinstance(block, dict) and block.get("block_public_acls") is False:
            return "exposed"
        return "protected"
    if rtype == "aws_s3_bucket_public_access_block":
        value = config.get("block_public_acls")
        if value is None:
            return "unclassified"
        return "exposed" if value is False else "protected"
    return "unclassified"


def evaluated_keys(res):
    """The keys the check actually read to reach its verdict, per resource type."""
    if res["type"] == "aws_s3_bucket_public_access_block":
        return ["block_public_acls"]
    keys = []
    if res["config"].get("acl") not in (None, "private"):
        keys.append("acl")
    if res["config"].get("public_access_block", {}).get("block_public_acls") is False:
        keys.append("public_access_block.block_public_acls")
    return keys


def strategy_c(resources):
    """Python check: one check, three-state classification, findings on
    'exposed' only.

    Returns {address: [evaluated keys]} for findings, matching the other
    strategies so the corpus can be scored uniformly.
    """
    hits = {}
    for res in resources:
        if classify(res) == "exposed":
            hits[res["address"]] = evaluated_keys(res)
    return hits


findings = strategy_c(FIXTURES)
for address, keys in findings.items():
    print(f"  FAIL {address:52s} evaluated {keys}")

print()
tally = {}
for res in FIXTURES:
    state = classify(res)
    tally[state] = tally.get(state, 0) + 1
print("classification:", ", ".join(f"{k}={v}" for k, v in sorted(tally.items())))
print("unclassified   :", [r["address"] for r in FIXTURES if classify(r) == "unclassified"])

## Step 6 — Score all four strategies

Recall is the share of genuinely exposed resources that were reported; precision
is the share of reports that were genuinely exposed. The false-positive count is
listed separately because it is the number a policy team actually feels — a
strategy with perfect recall and three false positives blocks a merge that did
not need blocking.

In [ ]:
def score(name, hits):
    flagged = set(hits)
    true_positive = len(flagged & EXPOSED)
    false_positive = len(flagged & COMPLIANT)
    return {
        "strategy": name,
        "flagged": len(flagged),
        "true_positives": true_positive,
        "false_positives": false_positive,
        "recall": true_positive / len(EXPOSED),
        "precision": true_positive / len(flagged) if flagged else 0.0,
    }


RESULTS = [
    score("A   single-attribute policy", strategy_a(FIXTURES)),
    score("B   whole-value policy set", strategy_b(FIXTURES)),
    score("B'  per-key policy set", strategy_b_prime(FIXTURES)),
    score("C   python check", strategy_c(FIXTURES)),
]

header = f"{'strategy':<30} {'flagged':>7} {'TP':>4} {'FP':>4} {'recall':>7} {'precision':>10}"
print(header)
print("-" * len(header))
for row in RESULTS:
    print(f"{row['strategy']:<30} {row['flagged']:>7} {row['true_positives']:>4} "
          f"{row['false_positives']:>4} {row['recall']:>6.0%} {row['precision']:>9.0%}")

### Reading the table

- **A** reaches 40% recall and no false positives. It is not wrong about what it
  covers; it is silent about the other two surfaces. The three exposed resources
  it misses are the ones that express their ACL state through the public-access
  block.
- **B** reaches full recall and pays for it with three false positives — the two
  partially-configured buckets and the standalone resource that relaxes the
  policy surface while leaving the ACL surface locked. All three are safe, and
  all three would be reported.
- **B′** matches C on this corpus: full recall, no false positives. The
  difference between them is not visible here, and the reason is worth stating
  plainly — the corpus enumerates exactly the surfaces both of them cover. The
  gap opens the day a surface is added to Terraform and neither policy is
  updated, and a finite corpus cannot demonstrate that.
- **C** ties B′ on findings and adds the `unclassified` bucket, which the binary
  strategies fold silently into "pass".

The practical conclusion is not that B′ is wrong. It is that the choice between
B′ and C is a choice about maintenance envelope and reporting, and it should be
made on that basis rather than on the score column.

## Step 7 — Verify the split

If a later edit changes what any strategy considers a finding, the assertions
below fail instead of producing a quietly different table. `exposed` and
`protected` are mutually exclusive and jointly exhaustive for the resources that
carry ACL information; `aws_s3_bucket.bare` is the one resource that carries
none, and is classified `unclassified` rather than being forced into either.

In [ ]:
expected_a = {
    "aws_s3_bucket.logs",
    "aws_s3_bucket.legacy_site",
}
expected_surfaces = expected_a | {
    "aws_s3_bucket.inline_block",
    "aws_s3_bucket_public_access_block.acls_open",
    "aws_s3_bucket_public_access_block.all_open",
}
expected_false_positives = {
    "aws_s3_bucket.partial_block",
    "aws_s3_bucket.unrelated_block_keys",
    "aws_s3_bucket_public_access_block.policies_open",
}

assert set(strategy_a(FIXTURES)) == expected_a, "A drifted from the single-attribute rule"
assert set(strategy_b(FIXTURES)) - EXPOSED == expected_false_positives, "B false positives drifted"
assert set(strategy_b_prime(FIXTURES)) == expected_surfaces, "B' no longer covers every surface"
assert set(strategy_c(FIXTURES)) == expected_surfaces, "C no longer covers every surface"
assert classify({"type": "aws_s3_bucket", "config": {}}) == "unclassified"
assert classify({"type": "aws_s3_bucket", "config": {"acl": "private"}}) == "protected"
assert all(
    state in ("exposed", "protected", "unclassified")
    for state in (classify(f) for f in FIXTURES)
)
print("verify: OK — all four strategies match the documented split")

## Step 8 — Hand the same policies to Checkov

Every strategy contributes its own check ID: `CKV_CUSTOM_ACL` for the
single-attribute policy, `CKV_CUSTOM_ACL_WHOLE` for the whole-value set,
`CKV_CUSTOM_ACL_PERKEY` for the per-key set, and `CVN_CUSTOM_ACL_CHECK` for the
Python check. The YAML policies and the Python module are both loaded from one
directory with `--external-checks-dir`, so a single scan exercises all four, and
the check ID in the output names the strategy that produced a given finding —
which is what makes a mixed set debuggable.

```bash
checkov -d ./infra \
  --framework terraform \
  --external-checks-dir ./checkov-policies \
  --check CKV_CUSTOM_ACL,CKV_CUSTOM_ACL_WHOLE,CKV_CUSTOM_ACL_PERKEY,CVN_CUSTOM_ACL_CHECK \
  --output json --compact
```

The Python form is the usual extension point — a check class whose
`scan_resource_conf` returns `CheckResult.PASSED` or `CheckResult.FAILED`
together with the list of keys it read. Unlike the YAML policies it can compute
the verdict from more than one attribute. It also receives parsed HCL values
rather than plain Python scalars: an attribute arrives wrapped in a list, and a
nested block arrives as a list of one block, which is why the comparisons below
index into `acl` and `block[0]`. The harness in the earlier steps uses plain
values; the decision rule is the same either way, only the unwrapping differs.

```python
from checkov.common.models.enums import CheckCategories, CheckResult
from checkov.terraform.checks.resource.base_resource_check import BaseResourceCheck


class S3BucketHasNoPublicAcl(BaseResourceCheck):
    def __init__(self):
        super().__init__(
            name="S3 bucket must not carry a public ACL",
            id="CVN_CUSTOM_ACL_CHECK",
            categories=[CheckCategories.GENERAL_SECURITY],
            supported_resources=["aws_s3_bucket", "aws_s3_bucket_public_access_block"],
        )

    def scan_resource_conf(self, conf):
        acl = conf.get("acl")
        if acl is not None and acl[0] != "private":
            return CheckResult.FAILED, ["acl"]

        block = conf.get("public_access_block")
        if isinstance(block, list) and block[0].get("block_public_acls") is False:
            return CheckResult.FAILED, ["public_access_block.block_public_acls"]

        # A standalone public-access-block resource carries the flag at the top
        # level, so it needs no block unwrapping.
        if conf.get("block_public_acls") == [False]:
            return CheckResult.FAILED, ["block_public_acls"]

        return CheckResult.PASSED
```

The check ID prefix is the only thing that separates a custom check from a
built-in one in the output, so pick a prefix that will not collide and keep it
stable once findings are being tracked against it.

## Common errors

- **Comparing a whole block against a literal.** This is Strategy B, and it is
  the most common shape to reach for because it is a single comparison. It
  reports every partially-configured resource. Compare the key that governs the
  decision, not the shape of the object that contains it.
- **Treating an absent attribute as a pass without saying so.** For this rule an
  absent ACL attribute is genuinely not a finding, but only because the provider
  default is private. A rule where the default is unsafe inverts this, and a
  silent `None → pass` becomes a silent hole.
- **Enumerating surfaces in YAML and assuming the list is complete.** B′ matches
  C on this corpus only because the corpus was written by the same person who
  wrote the policies. A surface added to Terraform is a pass, not a finding,
  until someone remembers the policy.
- **One check ID shared by several policies.** Findings tracked against a check
  ID become unattributable the moment two policies answer to the same ID. Give
  each policy its own, and keep the prefix stable once the ID is in a dashboard.
- **Ignoring the standalone resource type.** A team that writes the inline
  `public_access_block` policy and stops will pass every bucket whose block lives
  in a separate resource. The two surfaces are separate in Terraform and need
  separate policies or one check that covers both.

## Summary

For a single-surface rule, the single-attribute YAML policy at
`../policies/no-public-s3-buckets/no_public_s3_buckets.yaml` is the right
answer: it is one comparison, it is exact, and it has no false positives. Its one
weakness is that it says nothing about surfaces it does not name.

For a multi-surface rule, the choice is between a per-key policy set and a Python
check. Both achieve full recall on a corpus that matches their own enumeration.
The per-key set keeps the policy declarative and reviewable by someone who does
not write Python, at the cost of being a static list that must be maintained as
Terraform's surfaces change. The Python check centralises the surface list in one
place and can classify a resource it cannot evaluate as `unclassified` rather
than passing it silently, at the cost of being code that has to be tested and
kept current.

The score table is a tie between those two by construction. The decision belongs
to whoever owns the maintenance, and the `unclassified` bucket is the argument
for the Python form whenever a team needs to report on the gap rather than on the
findings.

## References

- `../policies/no-public-s3-buckets/no_public_s3_buckets.yaml` — the
  single-attribute policy this notebook generalises.
- `../snippets/terraform-scan-custom-policies.py` — running a scan with an
  external checks directory and gating on the exit code.
- `compare-builtin-vs-custom-k8s.ipynb` — the other decision, made before this
  one: whether a custom check is needed at all.
- `compare-static-vs-plan-scanning.ipynb` — scan-mode comparison, which
  determines what shape the resources reach a custom policy in.